# 01 · EDA — WC 2026 predictor
Overview of the inputs: teams, Matchday-1 results/form, and the historical match record.
Run `python -m src.ingest` first (or just run the next cell).

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import ingest, config as C, features as F
data = ingest.run()
teams, hist, md1 = data['teams'], data['historical'], data['md1']
print('teams', teams.shape, '| historical', hist.shape, '| md1', md1.shape)

## Teams by confederation & strength

In [ ]:
display(teams.sort_values('elo', ascending=False).head(10))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
teams['confederation'].value_counts().plot.bar(ax=ax[0], title='Teams per confederation')
sns.histplot(teams['elo'], bins=15, ax=ax[1]); ax[1].set_title('Elo distribution')
plt.tight_layout(); plt.show()

## Matchday-1 results & derived form

In [ ]:
md1['outcome'] = [F.outcome_label(h, a) for h, a in zip(md1.home_goals, md1.away_goals)]
display(md1)
form = F.team_form_from_results(md1)
form_df = pd.DataFrame(form).T.sort_values('pts', ascending=False)
form_df.columns = ['md1_goals_scored', 'md1_goals_conceded', 'md1_points']
display(form_df.head(12))

In [ ]:
ax = md1['outcome'].value_counts().plot.bar(title='MD1 outcome balance (home perspective)')
plt.show()

## Historical record (1930-2022)

In [ ]:
hist['total_goals'] = hist.home_goals + hist.away_goals
print('matches:', len(hist), '| avg goals/match:', round(hist.total_goals.mean(), 2))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
hist['year'].value_counts().sort_index().plot.bar(ax=ax[0], title='Matches per edition')
sns.histplot(hist['total_goals'], bins=range(0, 11), ax=ax[1]); ax[1].set_title('Goals per match')
plt.tight_layout(); plt.show()